In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="transformers",
    notebook_name="02_encoder_vs_decoder.ipynb"
)

# Encoder vs Decoder: Same Task, Different Architectures

We will build **two** transformer classifiers — one encoder-only (bidirectional, like BERT) and one decoder-only (causal, like GPT) — and train both on the same sentiment classification task.

Same data. Same model size. Same training setup. The only difference: **attention direction**.

By the end of this notebook, you will see which architecture works better for understanding, and *why*, by looking at the attention patterns.

**Prerequisites:** [Encoder vs Decoder (theory)](./encoder-vs-decoder.md) and [Training a Small Transformer](./01_training_a_small_transformer.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print(f"Using device: {device}")
print("Setup complete!")

## Part 1: The Sentiment Dataset

We use a small inline dataset of short sentences labeled as positive (1) or negative (0). No file downloads needed — everything lives right here in the notebook.

We use simple word-level tokenization: split on spaces, build a vocabulary.

In [ ]:
# Inline sentiment dataset
raw_data = [
    # Positive examples
    ("this movie was wonderful and amazing", 1),
    ("i love this place so much", 1),
    ("the food here is excellent", 1),
    ("what a great experience overall", 1),
    ("the service was outstanding today", 1),
    ("i really enjoyed every moment", 1),
    ("this is the best thing ever", 1),
    ("absolutely fantastic and highly recommended", 1),
    ("the quality is superb and impressive", 1),
    ("a truly delightful and fun time", 1),
    ("everything was perfect and lovely", 1),
    ("i am so happy and satisfied", 1),
    ("the team did an excellent job", 1),
    ("what a beautiful and pleasant day", 1),
    ("this product is amazing and useful", 1),
    ("i felt great after the visit", 1),
    ("the show was brilliant and exciting", 1),
    ("such a wonderful and kind gesture", 1),
    ("i highly recommend this to everyone", 1),
    ("the atmosphere was warm and welcoming", 1),
    ("a truly remarkable and positive experience", 1),
    ("the results were impressive and great", 1),
    ("i am very pleased with this", 1),
    ("the staff was friendly and helpful", 1),
    ("a fantastic meal with great flavors", 1),
    # Negative examples
    ("this movie was terrible and boring", 0),
    ("i hate this place so much", 0),
    ("the food here is awful today", 0),
    ("what a horrible experience overall", 0),
    ("the service was disappointing and slow", 0),
    ("i really disliked every moment here", 0),
    ("this is the worst thing ever", 0),
    ("absolutely dreadful and not recommended", 0),
    ("the quality is poor and lacking", 0),
    ("a truly unpleasant and bad time", 0),
    ("everything was broken and messy today", 0),
    ("i am so upset and frustrated", 0),
    ("the team did a terrible job", 0),
    ("what an ugly and miserable day", 0),
    ("this product is useless and fragile", 0),
    ("i felt awful after the visit", 0),
    ("the show was dull and painful", 0),
    ("such a wasteful and rude gesture", 0),
    ("i would never recommend this ever", 0),
    ("the atmosphere was cold and unwelcoming", 0),
    ("a truly regrettable and negative experience", 0),
    ("the results were disappointing and bad", 0),
    ("i am very annoyed with this", 0),
    ("the staff was rude and unhelpful", 0),
    ("a horrible meal with no flavor", 0),
]

print(f"Dataset size: {len(raw_data)} sentences")
print(f"  Positive: {sum(1 for _, l in raw_data if l == 1)}")
print(f"  Negative: {sum(1 for _, l in raw_data if l == 0)}")
print()
print("Examples:")
for text, label in raw_data[:3]:
    sentiment = "Positive" if label == 1 else "Negative"
    print(f"  [{sentiment}] {text}")

## Part 2: Tokenization and Padding

We build a simple word-level vocabulary by splitting sentences on spaces. Every sentence gets padded to the same length so we can process them in batches.

In [ ]:
# Build vocabulary from all words in the dataset
all_words = set()
for text, _ in raw_data:
    all_words.update(text.split())

# Special tokens: <PAD> for padding
word_to_idx = {"<PAD>": 0}
for i, word in enumerate(sorted(all_words), start=1):
    word_to_idx[word] = i

idx_to_word = {v: k for k, v in word_to_idx.items()}
vocab_size = len(word_to_idx)

# Find max sentence length for padding
max_len = max(len(text.split()) for text, _ in raw_data)

def encode_sentence(text, max_len):
    """Convert a sentence to padded token indices."""
    words = text.split()
    indices = [word_to_idx[w] for w in words]
    # Pad to max_len
    padding = [0] * (max_len - len(indices))
    return indices + padding, len(indices)

# Encode all data
encoded_data = []
labels = []
lengths = []

for text, label in raw_data:
    enc, length = encode_sentence(text, max_len)
    encoded_data.append(enc)
    labels.append(label)
    lengths.append(length)

X = torch.tensor(encoded_data, device=device)
Y = torch.tensor(labels, dtype=torch.float32, device=device)
L = torch.tensor(lengths, device=device)

print(f"Vocabulary size: {vocab_size} words (including <PAD>)")
print(f"Max sentence length: {max_len} words")
print(f"Tensor shapes: X={X.shape}, Y={Y.shape}")
print()
print("Example encoding:")
example_text = raw_data[0][0]
example_enc, example_len = encode_sentence(example_text, max_len)
print(f"  Text:    {example_text}")
print(f"  Indices: {example_enc}")
print(f"  Length:  {example_len} (padded to {max_len})")

## Part 3: Build the Shared Transformer Block

Both models use the same transformer block. The only difference will be:
- **Encoder:** no causal mask (bidirectional attention)
- **Decoder:** causal mask (left-to-right attention)

In [ ]:
class TransformerBlock(nn.Module):
    """Shared transformer block used by both encoder and decoder."""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # Sub-layer 1: attention + residual
        x_norm = self.ln1(x)
        attn_out, attn_weights = self.attn(
            x_norm, x_norm, x_norm,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=True
        )
        x = x + self.dropout(attn_out)
        
        # Sub-layer 2: FFN + residual
        x_norm = self.ln2(x)
        x = x + self.ffn(x_norm)
        
        return x, attn_weights

print("TransformerBlock defined — shared by both encoder and decoder.")

## Part 4: Encoder-Only Classifier

The encoder sees all words at once (bidirectional attention — no causal mask). For classification, we **mean-pool** all word representations into a single vector, then pass it through a linear layer to predict the label.

```
Words → Embedding + Position → Transformer Blocks (bidirectional) → Mean Pool → Linear → Prediction
```

In [ ]:
class EncoderClassifier(nn.Module):
    """Encoder-only (bidirectional) classifier. Like BERT."""
    
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(d_model)
        # Binary classification: single output neuron
        self.classifier = nn.Linear(d_model, 1)
    
    def forward(self, x, lengths, return_attention=False):
        batch_size, seq_len = x.shape
        
        # Embeddings
        positions = torch.arange(seq_len, device=x.device)
        x_emb = self.token_emb(x) + self.pos_emb(positions)
        x_emb = self.drop(x_emb)
        
        # Padding mask: True where padding exists
        padding_mask = (x == 0)
        
        # No causal mask — encoder sees everything!
        all_attn = []
        h = x_emb
        for block in self.blocks:
            h, attn_w = block(h, attn_mask=None, key_padding_mask=padding_mask)
            all_attn.append(attn_w)
        
        h = self.ln_f(h)
        
        # Mean pool over non-padding positions
        mask_expanded = (~padding_mask).unsqueeze(-1).float()  # (batch, seq, 1)
        pooled = (h * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1)
        
        logits = self.classifier(pooled).squeeze(-1)
        
        if return_attention:
            return logits, all_attn
        return logits

print("EncoderClassifier defined.")
print("  Attention: bidirectional (every word sees every word)")
print("  Classification: mean-pool all word vectors -> linear")

## Part 5: Decoder-Only Classifier

The decoder uses a **causal mask** — each word can only attend to itself and earlier words. For classification, we take the **last real token's** representation (the only position that has seen the entire sentence) and pass it through a linear layer.

```
Words → Embedding + Position → Transformer Blocks (causal) → Last Token → Linear → Prediction
```

In [ ]:
class DecoderClassifier(nn.Module):
    """Decoder-only (causal) classifier. Like GPT for classification."""
    
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 1)
    
    def forward(self, x, lengths, return_attention=False):
        batch_size, seq_len = x.shape
        
        # Embeddings
        positions = torch.arange(seq_len, device=x.device)
        x_emb = self.token_emb(x) + self.pos_emb(positions)
        x_emb = self.drop(x_emb)
        
        # Causal mask: decoder can only look left
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        
        # Padding mask
        padding_mask = (x == 0)
        
        all_attn = []
        h = x_emb
        for block in self.blocks:
            h, attn_w = block(h, attn_mask=causal_mask, key_padding_mask=padding_mask)
            all_attn.append(attn_w)
        
        h = self.ln_f(h)
        
        # Take the LAST real token (not padding)
        # lengths[i] gives the number of real tokens in sentence i
        last_indices = (lengths - 1).long()
        last_hidden = h[torch.arange(batch_size, device=x.device), last_indices]
        
        logits = self.classifier(last_hidden).squeeze(-1)
        
        if return_attention:
            return logits, all_attn
        return logits

print("DecoderClassifier defined.")
print("  Attention: causal (each word sees only past words)")
print("  Classification: last token vector -> linear")

## Part 6: Create Both Models with the Same Hyperparameters

Fair comparison: same d_model, same n_heads, same n_layers, same d_ff. The only difference is the attention mask.

In [ ]:
# Shared hyperparameters
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

# Create both models
encoder_model = EncoderClassifier(
    vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout
).to(device)

decoder_model = DecoderClassifier(
    vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout
).to(device)

enc_params = sum(p.numel() for p in encoder_model.parameters())
dec_params = sum(p.numel() for p in decoder_model.parameters())

print(f"Model Comparison:")
print(f"{'':>25s} {'Encoder':>12s} {'Decoder':>12s}")
print(f"-" * 50)
print(f"{'d_model':>25s} {d_model:>12d} {d_model:>12d}")
print(f"{'n_heads':>25s} {n_heads:>12d} {n_heads:>12d}")
print(f"{'n_layers':>25s} {n_layers:>12d} {n_layers:>12d}")
print(f"{'d_ff':>25s} {d_ff:>12d} {d_ff:>12d}")
print(f"{'Parameters':>25s} {enc_params:>12,d} {dec_params:>12,d}")
print(f"{'Attention':>25s} {'bidirectional':>12s} {'causal':>12s}")
print(f"{'Classification':>25s} {'mean pool':>12s} {'last token':>12s}")

## Part 7: Train Both Models

We train both models for the same number of epochs with the same optimizer and learning rate. Since the dataset is small, we use the full dataset as each batch.

In [ ]:
def train_model(model, X, Y, L, n_epochs, lr=1e-3):
    """Train a classifier and return loss + accuracy history."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    
    loss_history = []
    acc_history = []
    
    model.train()
    for epoch in range(n_epochs):
        logits = model(X, L)
        loss = criterion(logits, Y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # Compute accuracy
        with torch.no_grad():
            preds = (torch.sigmoid(logits) > 0.5).float()
            accuracy = (preds == Y).float().mean().item()
        
        loss_history.append(loss.item())
        acc_history.append(accuracy)
    
    return loss_history, acc_history


n_epochs = 300
lr = 1e-3

print(f"Training both models for {n_epochs} epochs...")
print()

# Train encoder
torch.manual_seed(42)
encoder_model = EncoderClassifier(
    vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout
).to(device)
enc_losses, enc_accs = train_model(encoder_model, X, Y, L, n_epochs, lr)
print(f"Encoder — Final loss: {enc_losses[-1]:.4f}, Final accuracy: {enc_accs[-1]*100:.1f}%")

# Train decoder
torch.manual_seed(42)
decoder_model = DecoderClassifier(
    vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout
).to(device)
dec_losses, dec_accs = train_model(decoder_model, X, Y, L, n_epochs, lr)
print(f"Decoder — Final loss: {dec_losses[-1]:.4f}, Final accuracy: {dec_accs[-1]*100:.1f}%")

## Part 8: Compare Training Curves

Let's plot loss and accuracy side by side. The encoder should converge faster and reach higher accuracy because it can see the entire sentence at every position.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Loss curves
ax1.plot(enc_losses, color='steelblue', linewidth=2, label='Encoder (bidirectional)', alpha=0.8)
ax1.plot(dec_losses, color='coral', linewidth=2, label='Decoder (causal)', alpha=0.8)
ax1.set_xlabel('Epoch', fontsize=13)
ax1.set_ylabel('BCE Loss', fontsize=13)
ax1.set_title('Training Loss', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(enc_accs, color='steelblue', linewidth=2, label='Encoder (bidirectional)', alpha=0.8)
ax2.plot(dec_accs, color='coral', linewidth=2, label='Decoder (causal)', alpha=0.8)
ax2.set_xlabel('Epoch', fontsize=13)
ax2.set_ylabel('Accuracy', fontsize=13)
ax2.set_title('Training Accuracy', fontsize=14)
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('Encoder vs Decoder: Same Task, Same Size, Different Attention',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Final Results:")
print(f"  Encoder: loss={enc_losses[-1]:.4f}, accuracy={enc_accs[-1]*100:.1f}%")
print(f"  Decoder: loss={dec_losses[-1]:.4f}, accuracy={dec_accs[-1]*100:.1f}%")
print()
print("The encoder typically converges faster and/or reaches higher accuracy")
print("because bidirectional attention gives it access to the full sentence")
print("at every position — a structural advantage for understanding tasks.")

## Part 9: Attention Heatmaps — Bidirectional vs Causal

This is where the architectural difference becomes visible. We feed the same sentence into both models and plot the attention weights.

- **Encoder:** Full square matrix — every word attends to every word
- **Decoder:** Lower triangle only — each word can only see past words

In [ ]:
# Pick a test sentence
test_text = "the food here is excellent"
test_words = test_text.split()
test_enc, test_len = encode_sentence(test_text, max_len)
test_x = torch.tensor([test_enc], device=device)
test_l = torch.tensor([test_len], device=device)

# Get attention weights from both models
encoder_model.eval()
decoder_model.eval()

with torch.no_grad():
    _, enc_attn = encoder_model(test_x, test_l, return_attention=True)
    _, dec_attn = decoder_model(test_x, test_l, return_attention=True)

# Plot attention for layer 1 of each model
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

n_words = len(test_words)

# Encoder attention (layer 1)
enc_map = enc_attn[0][0, :n_words, :n_words].cpu().numpy()
im1 = axes[0].imshow(enc_map, cmap='Blues', aspect='auto', vmin=0)
axes[0].set_xticks(range(n_words))
axes[0].set_yticks(range(n_words))
axes[0].set_xticklabels(test_words, fontsize=11, rotation=45, ha='right')
axes[0].set_yticklabels(test_words, fontsize=11)
axes[0].set_title('Encoder (Bidirectional)\nLayer 1 Attention', fontsize=13)
axes[0].set_xlabel('Key (attending TO)', fontsize=11)
axes[0].set_ylabel('Query (attending FROM)', fontsize=11)
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Decoder attention (layer 1)
dec_map = dec_attn[0][0, :n_words, :n_words].cpu().numpy()
im2 = axes[1].imshow(dec_map, cmap='Oranges', aspect='auto', vmin=0)
axes[1].set_xticks(range(n_words))
axes[1].set_yticks(range(n_words))
axes[1].set_xticklabels(test_words, fontsize=11, rotation=45, ha='right')
axes[1].set_yticklabels(test_words, fontsize=11)
axes[1].set_title('Decoder (Causal)\nLayer 1 Attention', fontsize=13)
axes[1].set_xlabel('Key (attending TO)', fontsize=11)
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.suptitle(f'Attention Patterns for "{test_text}"',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key differences:")
print("  Encoder: full attention matrix — every word can attend to every word")
print("  Decoder: triangular pattern — each word can only see itself and past words")
print()
print("For classification, the encoder has a structural advantage:")
print("  'the' can see 'excellent' (the sentiment word) directly.")
print("  In the decoder, 'the' cannot see 'excellent' — it only looks left.")

## Part 10: Layer 2 Attention Comparison

Let's also compare the second layer. In deeper layers, the decoder compensates for the causal mask by building up context through the residual stream. But the encoder still has the advantage of direct connections.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Encoder attention (layer 2)
enc_map2 = enc_attn[1][0, :n_words, :n_words].cpu().numpy()
im1 = axes[0].imshow(enc_map2, cmap='Blues', aspect='auto', vmin=0)
axes[0].set_xticks(range(n_words))
axes[0].set_yticks(range(n_words))
axes[0].set_xticklabels(test_words, fontsize=11, rotation=45, ha='right')
axes[0].set_yticklabels(test_words, fontsize=11)
axes[0].set_title('Encoder (Bidirectional)\nLayer 2 Attention', fontsize=13)
axes[0].set_xlabel('Key (attending TO)', fontsize=11)
axes[0].set_ylabel('Query (attending FROM)', fontsize=11)
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Decoder attention (layer 2)
dec_map2 = dec_attn[1][0, :n_words, :n_words].cpu().numpy()
im2 = axes[1].imshow(dec_map2, cmap='Oranges', aspect='auto', vmin=0)
axes[1].set_xticks(range(n_words))
axes[1].set_yticks(range(n_words))
axes[1].set_xticklabels(test_words, fontsize=11, rotation=45, ha='right')
axes[1].set_yticklabels(test_words, fontsize=11)
axes[1].set_title('Decoder (Causal)\nLayer 2 Attention', fontsize=13)
axes[1].set_xlabel('Key (attending TO)', fontsize=11)
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.suptitle(f'Layer 2 Attention Patterns for "{test_text}"',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("In layer 2, the decoder's last word has seen more context")
print("(indirectly, through the residual stream from layer 1).")
print("But the encoder still has a direct path from every word to every word.")

## Part 11: Why Decoder-Only Models Still Won the Scaling Race

You just saw that encoders have a structural advantage for understanding tasks. So why did the field move to decoder-only models (GPT, LLaMA, Claude) for almost everything, including classification?

Three reasons:

1. **Decoders can do generation AND understanding.** An encoder can classify text, but it cannot generate it. A decoder can do both. One architecture for all tasks is simpler to scale.

2. **Next-token prediction is a universal training objective.** You just need text — no labels, no task-specific data. The internet has trillions of tokens of unlabeled text. This is much more data than any labeled classification dataset.

3. **At large scale, the decoder's disadvantage shrinks.** With 2 layers, the encoder's bidirectional advantage is significant. With 96 layers, the decoder has many layers to propagate information from early tokens to later ones through the residual stream. The causal constraint matters less when the model is deep enough.

The takeaway: **for small models and understanding tasks, encoders win. For large models trained on massive data, decoders win because they can do everything.**

In [ ]:
# Final comparison summary
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)
print(f"")
print(f"{'Metric':<25s} {'Encoder':>15s} {'Decoder':>15s}")
print(f"-" * 55)
print(f"{'Final Loss':<25s} {enc_losses[-1]:>15.4f} {dec_losses[-1]:>15.4f}")
print(f"{'Final Accuracy':<25s} {enc_accs[-1]*100:>14.1f}% {dec_accs[-1]*100:>14.1f}%")

# Find epoch where each model first reached 90% accuracy
enc_90 = next((i for i, a in enumerate(enc_accs) if a >= 0.9), None)
dec_90 = next((i for i, a in enumerate(dec_accs) if a >= 0.9), None)
enc_90_str = str(enc_90) if enc_90 is not None else "never"
dec_90_str = str(dec_90) if dec_90 is not None else "never"
print(f"{'Epochs to 90% accuracy':<25s} {enc_90_str:>15s} {dec_90_str:>15s}")

print(f"{'Attention type':<25s} {'bidirectional':>15s} {'causal':>15s}")
print(f"{'Classification method':<25s} {'mean pool':>15s} {'last token':>15s}")
print(f"")
print("Key insight: same model size, same data, same training.")
print("The only difference is the attention mask — and it matters.")

## Summary

We built and trained two transformer classifiers on the same sentiment task:

| | Encoder (BERT-style) | Decoder (GPT-style) |
|---|---|---|
| Attention | Bidirectional | Causal (left-to-right) |
| Each word sees | All words | Only past words |
| Classification | Mean pool all positions | Last token only |
| Best for | Understanding tasks | Generation tasks |

### What we learned

1. **Encoders have a structural advantage for classification** because every word can see every other word, including sentiment-bearing words at any position.

2. **The causal mask creates an information bottleneck** in decoders — early words cannot see later words, so the last token must carry all the information.

3. **Attention heatmaps make the difference visible** — the encoder's full matrix vs the decoder's triangle show exactly what each architecture can and cannot see.

4. **Despite this, decoder-only models dominate in practice** because they can generate text, train on unlabeled data, and the bidirectional advantage shrinks at large scale.

---

[Previous: Training a Small Transformer](./01_training_a_small_transformer.ipynb) | [Back to Experiments Overview](./README.md)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)